In [ ]:
!pip install arch --quiet

import yfinance as yf
import pandas as pd
import numpy as np
from arch import arch_model
from google.colab import drive
import os

data_start_date = '2019-11-20'   # 抓資料的起始日（提早為方便滾動計算）
analysis_start_date = '2020-01-01'  # 分析輸出的起始日
end_date = '2025-08-10'

# 台灣市值前30名上市公司
tickers = [
    '2330.TW', '2317.TW', '2454.TW', '2308.TW', '2382.TW',
    '2881.TW', '2891.TW', '2882.TW', '2303.TW', '2412.TW',
    '2884.TW', '2886.TW', '3711.TW', '2357.TW', '1216.TW',
    '2885.TW', '2345.TW', '3231.TW', '3034.TW', '2892.TW',
    '2379.TW', '6669.TW', '2890.TW', '5880.TW', '2880.TW',
    '2383.TW', '3661.TW', '3017.TW', '2883.TW', '3008.TW'
]

drive.mount('/content/drive')

# 下載股價資料
data = yf.download(tickers, start=data_start_date, end=end_date, auto_adjust=False)

volume = data['Volume']
close_price = data['Close']
open_price = data['Open']
high_price = data['High']
low_price = data['Low']

# 計算成交值
turnover_value = volume * close_price

# 計算日報酬率
simple_return_open = open_price.pct_change()
simple_return_high = high_price.pct_change()
simple_return_low = low_price.pct_change()
simple_return_close = close_price.pct_change()

# Volume 和 Turnover 的報酬率
volume_return = volume.pct_change()
turnover_return = turnover_value.pct_change()

# 30 日年化歷史波動率
historical_volatility = (simple_return_close.rolling(window=30).std() * np.sqrt(252))

# GARCH 波動率計算
scaled_returns = simple_return_close * 100
garch_volatility = {}
for t in tickers:
    print(f"Fitting GARCH for {t}...")
    r = scaled_returns[t].dropna()
    model = arch_model(r, vol='Garch', p=1, q=1)
    res = model.fit(disp='off')
    garch_volatility[t] = res.conditional_volatility
garch_volatility_df = pd.DataFrame(garch_volatility).reindex(historical_volatility.index)

# 合併所有指標
combined_data = pd.DataFrame({
    **{f'VolumeReturn_{t}': volume_return[t] for t in tickers},
    **{f'TurnoverReturn_{t}': turnover_return[t] for t in tickers},
    **{f'SimpleReturn_Open_{t}': simple_return_open[t] for t in tickers},
    **{f'SimpleReturn_High_{t}': simple_return_high[t] for t in tickers},
    **{f'SimpleReturn_Low_{t}': simple_return_low[t] for t in tickers},
    **{f'SimpleReturn_Close_{t}': simple_return_close[t] for t in tickers},
    **{f'HistoricalVolatility_{t}': historical_volatility[t] for t in tickers},
    **{f'GARCHVolatility_{t}': garch_volatility_df[t] for t in tickers}
})

# 刪掉滾動計算的空值
combined_data = combined_data.iloc[29:]

# 篩選輸出日期
combined_data = combined_data[combined_data.index >= analysis_start_date]

# 輸出資料夾
output_folder = '/content/drive/MyDrive/Alina/tw_stock_top30/'
os.makedirs(output_folder, exist_ok=True)

# 儲存整合後資料
combined_data.to_csv(os.path.join(output_folder, 'top30_tw_stock_data.csv'), index=True)

# 分別存每支股票資料
for t in tickers:
    df = pd.DataFrame({
        'Date': combined_data.index,
        'VolumeReturn': combined_data[f'VolumeReturn_{t}'],
        'TurnoverReturn': combined_data[f'TurnoverReturn_{t}'],
        'SimpleReturn_Open': combined_data[f'SimpleReturn_Open_{t}'],
        'SimpleReturn_High': combined_data[f'SimpleReturn_High_{t}'],
        'SimpleReturn_Low': combined_data[f'SimpleReturn_Low_{t}'],
        'SimpleReturn_Close': combined_data[f'SimpleReturn_Close_{t}'],
        'HistoricalVolatility': combined_data[f'HistoricalVolatility_{t}'],
        'GARCHVolatility': combined_data[f'GARCHVolatility_{t}']
    })
    df.to_csv(os.path.join(output_folder, f'{t}_data.csv'), index=False)

print("資料已儲存到 Google Drive")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.3/985.3 kB 15.7 MB/s eta 0:00:00
Mounted at /content/drive


[*********************100%***********************]  30 of 30 completed


Fitting GARCH for 2330.TW...
Fitting GARCH for 2317.TW...
Fitting GARCH for 2454.TW...
Fitting GARCH for 2308.TW...
Fitting GARCH for 2382.TW...
Fitting GARCH for 2881.TW...
Fitting GARCH for 2891.TW...
Fitting GARCH for 2882.TW...
Fitting GARCH for 2303.TW...
Fitting GARCH for 2412.TW...
Fitting GARCH for 2884.TW...
Fitting GARCH for 2886.TW...
Fitting GARCH for 3711.TW...
Fitting GARCH for 2357.TW...
Fitting GARCH for 1216.TW...
Fitting GARCH for 2885.TW...
Fitting GARCH for 2345.TW...
Fitting GARCH for 3231.TW...
Fitting GARCH for 3034.TW...
Fitting GARCH for 2892.TW...
Fitting GARCH for 2379.TW...
Fitting GARCH for 6669.TW...
Fitting GARCH for 2890.TW...
Fitting GARCH for 5880.TW...
Fitting GARCH for 2880.TW...
Fitting GARCH for 2383.TW...
Fitting GARCH for 3661.TW...
Fitting GARCH for 3017.TW...
Fitting GARCH for 2883.TW...
Fitting GARCH for 3008.TW...
資料已儲存到 Google Drive


# 程式碼說明

**安裝與匯入套件**
   - 使用 `!pip install arch --quiet` 安裝 `arch` 套件，用於 GARCH 波動率模型計算。  
   - 匯入股票資料抓取 (`yfinance`)、資料處理 (`pandas`, `numpy`)、GARCH 模型 (`arch_model`)

**設定資料日期與股票清單**  
   - `data_start_date` 設定資料起始日（比分析日提前以方便滾動計算）。  
   - `analysis_start_date` 是最後分析輸出的起始日期。  
   - `end_date` 為資料擷取截止日期。  
   - `tickers` 為台灣市值前 30 大上市公司股票代碼清單。

**下載股價資料**  
   - 使用 `yf.download()` 取得多支股票從 `data_start_date` 到 `end_date` 的原始股價、成交量等資料。

**計算成交值與日報酬率**  
   - 透過成交量和收盤價計算每日成交值。  
   - 計算「開、高、低、收」價的每日簡單報酬率。（做return)  
   - 同時計算成交量和成交值的報酬率。

**計算歷史波動率**  
   - 用 30 日滾動視窗計算每日收盤報酬率的標準差，再年化（乘上√252）。

**GARCH(1,1) 模型波動率計算**  
   - 將收盤報酬率乘以100以符合 GARCH 模型數值尺度。  
   - 針對每支股票擬合 GARCH(1,1) 模型並取出模型估計的條件波動率。

**整合所有指標資料**  
   - 把成交量報酬率、成交值報酬率、各價位報酬率、歷史波動率和 GARCH 波動率全部合併成一個 DataFrame。

**資料清理與篩選**  
   - 刪除因滾動計算而產生的前 29 筆空值。  
   - 篩選出分析起始日之後的資料。

**儲存資料**  
- 在 Google Drive 指定資料夾建立路徑。  
- 把整合後的資料存成 CSV，並且每支股票個別存一個 CSV，方便後續分析使用。


## 波動率計算說明

### 30 日年化歷史波動率
- **historical_volatility = (simple_return_close.rolling(window=30).std() * np.sqrt(252))**
- 這段程式碼會計算過去30天的股價波動幅度，用最近30天股價變化的標準差來衡量股價「晃動」的大小。  
- 接著乘上一個換算因子（√252），把波動率換算成一年（大約有252個交易日）的波動幅度，方便跟不同時間長度的波動率做比較。

### GARCH(1,1) 波動率模型
- GARCH是一種用來「動態估計波動率」的統計模型，能夠捕捉波動率隨時間變化的特性。  
- 這裡先把報酬率放大100倍，是為了讓模型運算時數值更穩定。  
- 程式對每支股票分別套用 GARCH 模型，算出每天的「條件波動率」，它代表當天預測的波動風險大小，更能反映股價實際的波動變化。

---

總結來說：  
- **歷史波動率** 就是過去實際的波動幅度（用標準差衡量）  
- **GARCH 波動率** 則是透過模型預測、隨時間調整的動態波動率  
